# YOLO v11 Progressive Unfreezing Training for Building Damage Detection

This notebook implements advanced transfer learning to improve YOLO performance from 27.1% to 45-55% mAP@50.

## Key Techniques:
1. **Progressive Unfreezing**: Gradually unfreeze layers during training
2. **Discriminative Learning Rates**: Different LR for different layers
3. **One-Cycle Learning**: Optimal convergence
4. **Building-Specific Augmentations**: Domain-specific data augmentation
5. **Test Time Augmentation**: Improve inference accuracy

In [ ]:
# Setup environment
!pip install ultralytics==8.0.200 -q
!pip install albumentations wandb -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
!nvidia-smi

In [ ]:
import torch
import yaml
import json
import numpy as np
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
from ultralytics import YOLO
import wandb
from IPython.display import clear_output

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 1. Data Preparation

In [ ]:
# Upload dataset ZIP file
from google.colab import files
import zipfile

print("Please upload your dataset ZIP file...")
uploaded = files.upload()

# Extract dataset
for filename in uploaded.keys():
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('/content/dataset')
    print(f"Extracted {filename}")

# Verify dataset structure
!ls -la /content/dataset/

In [ ]:
# Analyze dataset quality
def analyze_dataset(yaml_path):
    """Analyze dataset for quality issues."""
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    print("Dataset Configuration:")
    print(f"Classes: {data['nc']}")
    print(f"Class names: {data['names']}")
    
    # Count annotations
    import glob
    train_labels = glob.glob(f"{data['path']}/{data['train']}/../labels/*.txt")
    val_labels = glob.glob(f"{data['path']}/{data['val']}/../labels/*.txt")
    
    # Class distribution
    class_counts = {name: 0 for name in data['names']}
    
    for label_file in train_labels[:100]:  # Sample first 100
        with open(label_file, 'r') as f:
            for line in f:
                class_id = int(line.split()[0])
                class_counts[data['names'][class_id]] += 1
    
    print(f"\nDataset Statistics:")
    print(f"Training images: {len(train_labels)}")
    print(f"Validation images: {len(val_labels)}")
    print(f"\nClass distribution (sample):")
    for name, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {name}: {count}")
    
    # Check for imbalance
    counts = list(class_counts.values())
    if max(counts) > 0 and min(counts) == 0:
        print("\n⚠️ Warning: Some classes have no samples!")
    elif max(counts) / (min(counts) + 1) > 10:
        print("\n⚠️ Warning: Severe class imbalance detected!")

analyze_dataset('/content/dataset/data.yaml')

## 2. Progressive Unfreezing Implementation

In [ ]:
class ProgressiveYOLOTrainer:
    """Progressive unfreezing trainer for YOLO."""
    
    def __init__(self, model_path='yolo11n.pt', data_yaml='/content/dataset/data.yaml'):
        self.model = YOLO(model_path)
        self.data_yaml = data_yaml
        self.best_map50 = 0
        self.stage_results = {}
        
        # Training stages
        self.stages = [
            {'name': 'head', 'epochs': 10, 'lr': 0.001, 'freeze': 20},
            {'name': 'neck', 'epochs': 10, 'lr': 0.0005, 'freeze': 15},
            {'name': 'backbone_late', 'epochs': 10, 'lr': 0.0002, 'freeze': 10},
            {'name': 'backbone_mid', 'epochs': 10, 'lr': 0.0001, 'freeze': 5},
            {'name': 'full_network', 'epochs': 20, 'lr': 0.00005, 'freeze': 0},
        ]
    
    def train_stage(self, stage):
        """Train a single stage."""
        print(f"\n{'='*50}")
        print(f"Stage: {stage['name']}")
        print(f"Epochs: {stage['epochs']}, LR: {stage['lr']}, Freeze: {stage['freeze']}")
        print(f"{'='*50}")
        
        # Train
        results = self.model.train(
            data=self.data_yaml,
            epochs=stage['epochs'],
            imgsz=640,
            batch=16,
            freeze=stage['freeze'],
            optimizer='AdamW',
            lr0=stage['lr'],
            lrf=0.01,
            momentum=0.937,
            weight_decay=0.0005,
            warmup_epochs=3,
            warmup_momentum=0.8,
            warmup_bias_lr=0.1,
            label_smoothing=0.1,
            patience=10,
            save=True,
            cache=True,
            amp=True,
            mosaic=1.0 if stage['name'] != 'full_network' else 0.5,
            mixup=0.2,
            copy_paste=0.1,
            degrees=15,
            translate=0.1,
            scale=0.5,
            fliplr=0.5,
            hsv_h=0.015,
            hsv_s=0.7,
            hsv_v=0.4,
            device=0,
            project='/content/runs',
            name=stage['name'],
            exist_ok=True
        )
        
        # Get metrics
        metrics = {
            'map50': results.results_dict.get('metrics/mAP50(B)', 0),
            'map50_95': results.results_dict.get('metrics/mAP50-95(B)', 0),
            'precision': results.results_dict.get('metrics/precision(B)', 0),
            'recall': results.results_dict.get('metrics/recall(B)', 0),
        }
        
        self.stage_results[stage['name']] = metrics
        
        # Save best model
        if metrics['map50'] > self.best_map50:
            self.best_map50 = metrics['map50']
            self.model.save('/content/best_model.pt')
            print(f"✅ New best model! mAP@50: {self.best_map50:.4f}")
        
        return metrics
    
    def run_progressive_training(self):
        """Run all training stages."""
        for i, stage in enumerate(self.stages, 1):
            print(f"\n{'#'*60}")
            print(f"STAGE {i}/{len(self.stages)}")
            print(f"{'#'*60}")
            
            metrics = self.train_stage(stage)
            
            # Clear output for cleaner display
            clear_output(wait=True)
            
            # Show progress
            self.show_progress(i)
        
        return self.best_map50
    
    def show_progress(self, current_stage):
        """Display training progress."""
        print("\n" + "="*60)
        print("PROGRESSIVE TRAINING PROGRESS")
        print("="*60)
        
        for i, (stage_name, metrics) in enumerate(self.stage_results.items(), 1):
            status = "✅" if i <= current_stage else "⏳"
            print(f"{status} Stage {i}: {stage_name}")
            if i <= current_stage:
                print(f"   mAP@50: {metrics['map50']:.4f}")
                print(f"   mAP@50-95: {metrics['map50_95']:.4f}")
                print(f"   Precision: {metrics['precision']:.4f}")
                print(f"   Recall: {metrics['recall']:.4f}")
        
        print(f"\nBest mAP@50: {self.best_map50:.4f}")
        
        # Progress bar
        progress = current_stage / len(self.stages)
        bar_length = 40
        filled = int(bar_length * progress)
        bar = '█' * filled + '░' * (bar_length - filled)
        print(f"\nProgress: [{bar}] {progress*100:.0f}%")

## 3. Start Training

In [ ]:
# Initialize WandB (optional)
wandb.init(
    project="yolo-building-damage",
    name=f"progressive-{datetime.now().strftime('%Y%m%d-%H%M')}",
    config={
        'architecture': 'YOLOv11n',
        'technique': 'progressive_unfreezing',
        'target_map50': 0.45
    }
)

# Create trainer
trainer = ProgressiveYOLOTrainer(
    model_path='yolo11n.pt',  # Download base model
    data_yaml='/content/dataset/data.yaml'
)

# Run progressive training
print("Starting Progressive Unfreezing Training...")
print("This will take several hours. Get some coffee! ☕\n")

final_map = trainer.run_progressive_training()

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print(f"Final mAP@50: {final_map:.4f}")

if final_map >= 0.45:
    print("✅ SUCCESS! Target performance achieved!")
else:
    print(f"⚠️ Below target. Achieved {final_map:.4f}, target was 0.45")

## 4. Advanced Techniques

In [ ]:
# Test Time Augmentation (TTA)
def apply_tta(model_path, data_yaml):
    """Apply TTA for better inference."""
    model = YOLO(model_path)
    
    # Run validation with augmentation
    results = model.val(
        data=data_yaml,
        augment=True,  # Enable TTA
        imgsz=640,
        batch=16
    )
    
    print("TTA Results:")
    print(f"mAP@50: {results.box.map50:.4f}")
    print(f"mAP@50-95: {results.box.map:.4f}")
    
    return results

print("Applying Test Time Augmentation...")
tta_results = apply_tta('/content/best_model.pt', '/content/dataset/data.yaml')

In [ ]:
# Pseudo-labeling for additional training data
def generate_pseudo_labels(model_path, unlabeled_images_path, confidence_threshold=0.7):
    """Generate pseudo-labels for unlabeled data."""
    model = YOLO(model_path)
    
    import glob
    import os
    
    images = glob.glob(f"{unlabeled_images_path}/*.jpg")
    pseudo_labels_path = '/content/pseudo_labels'
    os.makedirs(pseudo_labels_path, exist_ok=True)
    
    high_conf_count = 0
    
    for img_path in images:
        # Predict
        results = model(img_path)
        
        # Filter high confidence predictions
        for r in results:
            if r.boxes is not None:
                # Get high confidence detections
                conf_mask = r.boxes.conf > confidence_threshold
                
                if conf_mask.any():
                    high_conf_count += 1
                    
                    # Save pseudo-label in YOLO format
                    label_path = os.path.join(
                        pseudo_labels_path,
                        os.path.basename(img_path).replace('.jpg', '.txt')
                    )
                    
                    with open(label_path, 'w') as f:
                        boxes = r.boxes[conf_mask]
                        for box in boxes.xywhn:  # Normalized xywh format
                            cls = int(boxes.cls[0])
                            x, y, w, h = box
                            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")
    
    print(f"Generated {high_conf_count} high-confidence pseudo-labels")
    print(f"Saved to {pseudo_labels_path}")
    
    return pseudo_labels_path

# Example usage (uncomment if you have unlabeled data)
# pseudo_path = generate_pseudo_labels(
#     '/content/best_model.pt',
#     '/content/unlabeled_images',
#     confidence_threshold=0.8
# )

## 5. Model Evaluation & Export

In [ ]:
# Detailed evaluation
def evaluate_model(model_path, data_yaml):
    """Comprehensive model evaluation."""
    model = YOLO(model_path)
    
    # Validate
    results = model.val(data=data_yaml)
    
    # Per-class metrics
    print("\nPer-Class Performance:")
    print("-" * 50)
    
    with open(data_yaml, 'r') as f:
        data = yaml.safe_load(f)
    
    for i, class_name in enumerate(data['names']):
        if i < len(results.box.ap50):
            ap50 = results.box.ap50[i]
            print(f"{class_name:20} AP@50: {ap50:.4f}")
    
    # Overall metrics
    print("\nOverall Metrics:")
    print("-" * 50)
    print(f"mAP@50:      {results.box.map50:.4f}")
    print(f"mAP@50-95:   {results.box.map:.4f}")
    print(f"Precision:   {results.box.mp:.4f}")
    print(f"Recall:      {results.box.mr:.4f}")
    
    return results

print("Evaluating final model...")
eval_results = evaluate_model('/content/best_model.pt', '/content/dataset/data.yaml')

In [ ]:
# Export to ONNX for production
def export_to_onnx(model_path):
    """Export model to ONNX format."""
    model = YOLO(model_path)
    
    # Export
    onnx_path = model.export(format='onnx', opset=11, simplify=True)
    
    print(f"Model exported to: {onnx_path}")
    
    # Verify export
    import onnx
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("✅ ONNX model validated successfully")
    
    # Get model size
    import os
    size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"Model size: {size_mb:.2f} MB")
    
    return onnx_path

print("Exporting to ONNX...")
onnx_path = export_to_onnx('/content/best_model.pt')

In [ ]:
# Generate final report
def generate_final_report(trainer, eval_results):
    """Generate comprehensive training report."""
    report = {
        'timestamp': datetime.now().isoformat(),
        'best_map50': trainer.best_map50,
        'stage_results': trainer.stage_results,
        'final_evaluation': {
            'map50': eval_results.box.map50,
            'map50_95': eval_results.box.map,
            'precision': eval_results.box.mp,
            'recall': eval_results.box.mr
        },
        'recommendations': []
    }
    
    # Generate recommendations
    if trainer.best_map50 < 0.45:
        report['recommendations'].extend([
            "Increase dataset size with more diverse examples",
            "Try larger YOLO variant (yolo11s or yolo11m)",
            "Extend training epochs",
            "Adjust class weights for imbalanced classes"
        ])
    elif trainer.best_map50 < 0.55:
        report['recommendations'].extend([
            "Fine-tune hyperparameters",
            "Implement pseudo-labeling",
            "Try model ensemble"
        ])
    else:
        report['recommendations'].append("Excellent performance achieved!")
    
    # Save report
    with open('/content/training_report.json', 'w') as f:
        json.dump(report, f, indent=2)
    
    print("\n" + "="*60)
    print("FINAL TRAINING REPORT")
    print("="*60)
    print(f"Best mAP@50: {trainer.best_map50:.4f}")
    print(f"Target: 0.45")
    print(f"Status: {'✅ ACHIEVED' if trainer.best_map50 >= 0.45 else '⚠️ BELOW TARGET'}")
    print("\nRecommendations:")
    for rec in report['recommendations']:
        print(f"  - {rec}")
    print("\nReport saved to: /content/training_report.json")
    
    return report

report = generate_final_report(trainer, eval_results)

In [ ]:
# Download trained model and report
from google.colab import files

print("Preparing files for download...")

# Create zip with all important files
import zipfile
import os

with zipfile.ZipFile('/content/trained_model.zip', 'w') as zipf:
    zipf.write('/content/best_model.pt', 'best_model.pt')
    if os.path.exists(onnx_path):
        zipf.write(onnx_path, 'best_model.onnx')
    zipf.write('/content/training_report.json', 'training_report.json')

print("Downloading trained model package...")
files.download('/content/trained_model.zip')

print("\n✅ Training complete! Files downloaded.")
print("Upload the ONNX model to Supabase storage for production use.")